# Data Prep

* View data_prep folder and pipeline file for detailed info
* Parameter to change on new Data: constants.py with cols and paths,
also fail_on_duplcate must be set to true if duplicated values are viewed to be an error (timestamps are always errors).

In [23]:
from pathlib import Path
import sys

# Ensure the project src/ directory is importable from the notebook.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in [_cwd, *_cwd.parents] if (p / "src" / "timeseries_forecast").exists()
)
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from timeseries_forecast.data_prep.raw_data import read_raw_metals_data
from timeseries_forecast.data_prep.pipeline import run_data_pipeline
from timeseries_forecast.data_prep.data_merge import merge_all_raw_data
from timeseries_forecast.data_prep.constants import RawMetalsColumn

# from timeseries_forecast.feature import autoregressive_features
# from timeseries_forecast.eval.eval import evaluate, evaluate_predictions
# from timeseries_forecast.feature.custom_features import add_log_ret_features
# from timeseries_forecast.feature import engineer_features
# from timeseries_forecast.config.feature_config import FeatureConfig
# from timeseries_forecast.data_prep.data_resample import resample_dataframe_with_aggregations

In [24]:
alu_df = read_raw_metals_data("aluminium")
alu_df.Date

0      2024-08-29
1      2024-08-28
2      2024-08-27
3      2024-08-26
4      2024-08-23
          ...    
2730   2014-01-07
2731   2014-01-06
2732   2014-01-03
2733   2014-01-02
2734   2014-01-01
Name: Date, Length: 2735, dtype: datetime64[us]

## Cleaning for all metals with same parameters

### Aluminium

In [25]:
aluminium_df_cleaned = run_data_pipeline(
    read_raw_metals_data("aluminium"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[RawMetalsColumn.VOLUME],
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])
aluminium_df_cleaned


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
⚠ 4 Zeilen sind doppelt (identischer Inhalt)
  Beispiele (erste 5 Timestamps):
DatetimeIndex(['2020-09-29', '2020-09-30', '2024-07-30', '2024-07-31'], dtype='datetime64[us]', name='RawMetalsColumn.OPEN_TIME', freq=None)

Fehlende Werte Check
⚠ Es gibt 1159 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 15 NaN-Werte insgesamt gefunden:
  Vol.: 15 (0.55%)

0-Werte Check
⚠ 32 0-Werte insgesamt gefunden:
  Vol.: 32 (1.17%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,894 Einträge
  Aktueller DataFrame:    2,735 Zeilen
  Zu ergänzende Einträge: 1,159
    RawMetalsColumn.VOLUME: 32 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1159 → 0 NaN
    Open: 1159 → 0 NaN
    High: 1159 → 0 NaN
    Low: 1159 → 0 NaN
    Vol.: 1206 → 0 NaN
    Change %: 1159 → 0 NaN

  Shape vorher:  (2735, 6)
  Shape nachher: (3894, 6)
✓ Interpolation abgeschlossen!

Duplik

,Close,Open,High,Low,Vol.,Mode
2014-01-01,110.10,110.000000,110.700000,109.900000,450.000000,Train
2014-01-02,111.20,110.550000,112.200000,110.450000,4720.000000,Train
2014-01-03,109.20,110.900000,111.100000,109.050000,4520.000000,Train
2014-01-04,109.35,110.433333,110.650000,108.733333,4333.333333,Train
2014-01-05,109.50,109.966667,110.200000,108.416667,4146.666667,Train
...,...,...,...,...,...,...
2024-08-25,229.60,229.450000,232.433333,227.483333,1236.666667,Test
2024-08-26,229.50,230.850000,233.500000,228.850000,510.000000,Test
2024-08-27,231.55,229.000000,233.000000,228.200000,60.000000,Test
2024-08-28,232.05,233.400000,233.400000,231.400000,35.000000,Test


In [6]:
# alu_df_cleaned.loc["25-08-2024"]

### Copper

In [26]:
copper_df_cleaned = run_data_pipeline(
    read_raw_metals_data("copper"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[RawMetalsColumn.VOLUME],
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
⚠ Es gibt 1136 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 20 NaN-Werte insgesamt gefunden:
  Vol.: 20 (0.73%)

0-Werte Check
⚠ 5 0-Werte insgesamt gefunden:
  Vol.: 5 (0.18%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,894 Einträge
  Aktueller DataFrame:    2,758 Zeilen
  Zu ergänzende Einträge: 1,136
    RawMetalsColumn.VOLUME: 5 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1136 → 0 NaN
    Open: 1136 → 0 NaN
    High: 1136 → 0 NaN
    Low: 1136 → 0 NaN
    Vol.: 1161 → 0 NaN
    Change %: 1136 → 0 NaN

  Shape vorher:  (2758, 6)
  Shape nachher: (3894, 6)
✓ Interpolation abgeschlossen!

Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
✓ Keine Lücken in der Zeitreihe gefunden (Frequenz: D).

NaN-Ch

### gold

In [27]:
gold_df_cleaned = run_data_pipeline(
    read_raw_metals_data("gold"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[RawMetalsColumn.VOLUME],
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
⚠ Es gibt 1135 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 22 NaN-Werte insgesamt gefunden:
  Vol.: 22 (0.80%)

0-Werte Check
⚠ 7 0-Werte insgesamt gefunden:
  Vol.: 7 (0.25%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,895 Einträge
  Aktueller DataFrame:    2,760 Zeilen
  Zu ergänzende Einträge: 1,135
    RawMetalsColumn.VOLUME: 7 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1135 → 0 NaN
    Open: 1135 → 0 NaN
    High: 1135 → 0 NaN
    Low: 1135 → 0 NaN
    Vol.: 1164 → 0 NaN
    Change %: 1135 → 0 NaN

  Shape vorher:  (2760, 6)
  Shape nachher: (3895, 6)
✓ Interpolation abgeschlossen!

Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
✓ Keine Lücken in der Zeitreihe gefunden (Frequenz: D).

NaN-Ch

### lead

In [28]:
lead_df_cleaned = run_data_pipeline(
    read_raw_metals_data("lead"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[RawMetalsColumn.VOLUME],
    fail_on_row_duplicates=False,
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
⚠ 6 Zeilen sind doppelt (identischer Inhalt)
  Beispiele (erste 5 Timestamps):
DatetimeIndex(['2023-03-29', '2023-03-30', '2023-07-27', '2023-07-28',
               '2024-02-26'],
              dtype='datetime64[us]', name='RawMetalsColumn.OPEN_TIME', freq=None)

Fehlende Werte Check
⚠ Es gibt 1137 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 31 NaN-Werte insgesamt gefunden:
  Vol.: 31 (1.12%)

0-Werte Check
⚠ 45 0-Werte insgesamt gefunden:
  Vol.: 45 (1.63%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,894 Einträge
  Aktueller DataFrame:    2,757 Zeilen
  Zu ergänzende Einträge: 1,137
    RawMetalsColumn.VOLUME: 45 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1137 → 0 NaN
    Open: 1137 → 0 NaN
    High: 1137 → 0 NaN
    Low: 1137 → 0 NaN
    Vol.: 1213 → 0 NaN
    Change %: 1137 → 0 NaN

  Shape vorher:  (2757, 6)
  Shape nachher: (3894

### nickel

In [29]:
nickel_df_cleaned = run_data_pipeline(
    read_raw_metals_data("nickel"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[
        RawMetalsColumn.VOLUME,
        RawMetalsColumn.OPEN,
        RawMetalsColumn.HIGH,
        RawMetalsColumn.LOW,
    ],
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])
nickel_df_cleaned


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
⚠ 21 Zeilen sind doppelt (identischer Inhalt)
  Beispiele (erste 5 Timestamps):
DatetimeIndex(['2022-05-27', '2022-05-30', '2022-05-31', '2022-06-02',
               '2022-06-03'],
              dtype='datetime64[us]', name='RawMetalsColumn.OPEN_TIME', freq=None)

Fehlende Werte Check
⚠ Es gibt 1139 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 580 NaN-Werte insgesamt gefunden:
  Vol.: 580 (21.05%)

0-Werte Check
⚠ 40 0-Werte insgesamt gefunden:
  Open: 1 (0.04%)
  High: 1 (0.04%)
  Low: 1 (0.04%)
  Vol.: 37 (1.34%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,894 Einträge
  Aktueller DataFrame:    2,755 Zeilen
  Zu ergänzende Einträge: 1,139
    RawMetalsColumn.VOLUME: 37 Nullwerte → NaN ersetzt
    RawMetalsColumn.OPEN: 1 Nullwerte → NaN ersetzt
    RawMetalsColumn.HIGH: 1 Nullwerte → NaN ersetzt
    RawMetalsColumn.LOW: 1 Nullwerte → NaN ersetzt

  Interpoliere 6 

,Close,Open,High,Low,Vol.,Mode
2014-01-01,866.100000,867.000000,869.900000,863.000000,2430.000000,Train
2014-01-02,875.200000,869.900000,880.500000,862.600000,20390.000000,Train
2014-01-03,865.100000,873.100000,875.000000,863.500000,13910.000000,Train
2014-01-04,865.100000,865.100000,865.100000,865.100000,16163.333333,Train
2014-01-05,858.100000,866.300000,867.200000,856.100000,18416.666667,Train
...,...,...,...,...,...,...
2024-08-25,1415.833333,1415.833333,1415.833333,1415.833333,20.000000,Test
2024-08-26,1420.400000,1420.400000,1420.400000,1420.400000,20.000000,Test
2024-08-27,1420.400000,1420.400000,1420.400000,1420.400000,20.000000,Test
2024-08-28,1438.000000,1438.000000,1438.000000,1438.000000,20.000000,Test


### silver

In [30]:
silver_df_cleaned = run_data_pipeline(
    read_raw_metals_data("silver"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[
        RawMetalsColumn.VOLUME,
        RawMetalsColumn.OPEN,
        RawMetalsColumn.HIGH,
        RawMetalsColumn.LOW,
    ],
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
⚠ Es gibt 1136 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 18 NaN-Werte insgesamt gefunden:
  Vol.: 18 (0.65%)

0-Werte Check
⚠ 5 0-Werte insgesamt gefunden:
  Vol.: 5 (0.18%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,895 Einträge
  Aktueller DataFrame:    2,759 Zeilen
  Zu ergänzende Einträge: 1,136
    RawMetalsColumn.VOLUME: 5 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1136 → 0 NaN
    Open: 1136 → 0 NaN
    High: 1136 → 0 NaN
    Low: 1136 → 0 NaN
    Vol.: 1159 → 0 NaN
    Change %: 1136 → 0 NaN

  Shape vorher:  (2759, 6)
  Shape nachher: (3895, 6)
✓ Interpolation abgeschlossen!

Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
✓ Keine Lücken in der Zeitreihe gefunden (Frequenz: D).

NaN-Ch

### zinc

In [31]:
zinc_df_cleaned = run_data_pipeline(
    read_raw_metals_data("zinc"),
    timestamp_column=RawMetalsColumn.OPEN_TIME,
    data_freq="D",
    split_train_end="2019-12-31",
    split_val_end="2021-12-31",
    zero_value_ignore_cols=[RawMetalsColumn.PCT_CHANGE],
    zero_as_nan_cols=[RawMetalsColumn.VOLUME],
).rename(columns={RawMetalsColumn.PRICE: "Close"}).drop(columns=[RawMetalsColumn.PCT_CHANGE])


Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
⚠ Es gibt 1137 Lücken in der Zeitreihe (erwartete Frequenz: D).

NaN-Check
⚠ 19 NaN-Werte insgesamt gefunden:
  Vol.: 19 (0.69%)

0-Werte Check
⚠ 13 0-Werte insgesamt gefunden:
  Vol.: 13 (0.47%)

Data Cleaning – Interpolation von Missing Values und Nullwerten

  Vollständiger Zeitindex: 3,895 Einträge
  Aktueller DataFrame:    2,758 Zeilen
  Zu ergänzende Einträge: 1,137
    RawMetalsColumn.VOLUME: 13 Nullwerte → NaN ersetzt

  Interpoliere 6 numerische Spalten:
    Price: 1137 → 0 NaN
    Open: 1137 → 0 NaN
    High: 1137 → 0 NaN
    Low: 1137 → 0 NaN
    Vol.: 1169 → 0 NaN
    Change %: 1137 → 0 NaN

  Shape vorher:  (2758, 6)
  Shape nachher: (3895, 6)
✓ Interpolation abgeschlossen!

Duplikat-Check
✓ Keine doppelten Zeitstempel im Index
✓ Keine identischen Zeilenduplikate gefunden

Fehlende Werte Check
✓ Keine Lücken in der Zeitreihe gefunden (Frequenz: D).

NaN

In [32]:
merged_df = merge_all_raw_data(
    aluminium_df_cleaned=aluminium_df_cleaned,
    copper_df_cleaned=copper_df_cleaned,
    gold_df_cleaned=gold_df_cleaned,
    lead_df_cleaned=lead_df_cleaned,
    nickel_df_cleaned=nickel_df_cleaned,
    silver_df_cleaned=silver_df_cleaned,
    zinc_df_cleaned=zinc_df_cleaned,
)

In [33]:
df = merged_df.copy()
df.columns

Index(['Aluminium_Close', 'Aluminium_Open', 'Aluminium_High', 'Aluminium_Low',
       'Aluminium_Vol.', 'Aluminium_Mode', 'Copper_Close', 'Copper_Open',
       'Copper_High', 'Copper_Low', 'Copper_Vol.', 'Copper_Mode', 'Gold_Close',
       'Gold_Open', 'Gold_High', 'Gold_Low', 'Gold_Vol.', 'Gold_Mode',
       'Lead_Close', 'Lead_Open', 'Lead_High', 'Lead_Low', 'Lead_Vol.',
       'Lead_Mode', 'Nickel_Close', 'Nickel_Open', 'Nickel_High', 'Nickel_Low',
       'Nickel_Vol.', 'Nickel_Mode', 'Silver_Close', 'Silver_Open',
       'Silver_High', 'Silver_Low', 'Silver_Vol.', 'Silver_Mode', 'Zinc_Close',
       'Zinc_Open', 'Zinc_High', 'Zinc_Low', 'Zinc_Vol.', 'Zinc_Mode'],
      dtype='str')

In [ ]:
# # Test the function
# frequency_resample = '1h'

# resampled = resample_dataframe_with_aggregations(df, frequency_resample)
# print(f"Original shape: {df.shape}")
# print(f"Resampled {frequency_resample} shape: {resampled.shape}")
# print(f"\nFirst few rows of resampled data:")
# print(resampled.head())

Original shape: (70077, 28)
Resampled 1h shape: (70077, 28)

First few rows of resampled data:
                     BTC_Open  BTC_High   BTC_Low  BTC_Close   BTC_Volume  \
2018-01-02 09:00:00  13342.97  13523.46  13326.10   13452.00   511.542770   
2018-01-02 10:00:00  13460.00  13617.28  13401.60   13560.00   654.454591   
2018-01-02 11:00:00  13566.50  13573.00  13401.05   13490.00   684.541650   
2018-01-02 12:00:00  13490.00  13590.00  13450.46   13489.04   671.205943   
2018-01-02 13:00:00  13489.04  13875.00  13470.41   13864.98  1036.836330   

                     BTC_Quote asset volume  BTC_Number of trades  \
2018-01-02 09:00:00            6.875573e+06                4383.0   
2018-01-02 10:00:00            8.861014e+06                5114.0   
2018-01-02 11:00:00            9.232321e+06                4638.0   
2018-01-02 12:00:00            9.081599e+06                5411.0   
2018-01-02 13:00:00            1.420579e+07                9538.0   

                     BTC_Ta

In [ ]:
# df = resampled.copy()

In [ ]:
# models = [
#     # RandomForestRegressor(random_state=42),
#     # DecisionTreeRegressor(random_state=42),
#     SVR(),
#     LinearSVR(random_state=42),
#     LGBMRegressor(random_state=42),
#     XGBRegressor(random_state=42),
#     LinearRegression(),
#     KNeighborsRegressor()
# ]

In [ ]:
# fügt eine type spalte als label (target) hinzu
# macht die timestamp spalte explizit, damit man daraus infos extrahieren kann

df["type"] = "Aluminium"
df["timestamp"] = df.index

In [45]:
# add log ret features
from timeseries_forecast.feature.custom_features import add_log_ret_features

feature_df, feature_name_btc_log_ret = add_log_ret_features(df, "Aluminium_Close")
#feature_df, feature_name_eth_log_ret = add_log_ret_features(feature_df, "ETH_Close")

In [ ]:
# feature_df.columns

Index(['BTC_Open', 'BTC_High', 'BTC_Low', 'BTC_Close', 'BTC_Volume',
       'BTC_Quote asset volume', 'BTC_Number of trades',
       'BTC_Taker buy base asset volume', 'BTC_Taker buy quote asset volume',
       'BTC_Mode', 'ETH_Open', 'ETH_High', 'ETH_Low', 'ETH_Close',
       'ETH_Volume', 'ETH_Mode', 'GOLD_Open', 'GOLD_High', 'GOLD_Low',
       'GOLD_Close', 'GOLD_Volume', 'GOLD_Mode', 'SP500_Open', 'SP500_High',
       'SP500_Low', 'SP500_Close', 'SP500_Volume', 'SP500_Mode', 'type',
       'timestamp', 'BTC_Close_log_ret'],
      dtype='object')

In [37]:
#without mode

numeric_feature_columns = ['Aluminium_Close', 'Aluminium_Open', 'Aluminium_High', 'Aluminium_Low',
       'Aluminium_Vol.', 'Copper_Close', 'Copper_Open',
       'Copper_High', 'Copper_Low', 'Copper_Vol.', 'Gold_Close',
       'Gold_Open', 'Gold_High', 'Gold_Low', 'Gold_Vol.',
       'Lead_Close', 'Lead_Open', 'Lead_High', 'Lead_Low', 'Lead_Vol.',
       'Nickel_Close', 'Nickel_Open', 'Nickel_High', 'Nickel_Low',
       'Nickel_Vol.', 'Silver_Close', 'Silver_Open',
       'Silver_High', 'Silver_Low', 'Silver_Vol.', 'Zinc_Close',
       'Zinc_Open', 'Zinc_High', 'Zinc_Low', 'Zinc_Vol.'
       ]

# Feature Engineering

### Lag Features

In [39]:
from timeseries_forecast.feature import engineer_features
lag_window_len = 12

lag = list(range(1, lag_window_len + 1))

features = []
for column in numeric_feature_columns:
    features.append(
        engineer_features.LaggingFeatures
        (
            column=column, 
            lags=lag,
         
        )
    )
    

### Rolling Features

In [40]:
for column in numeric_feature_columns:
    features.append(
        engineer_features.RollingFeatures
        (
            column=column, 
            rolls=[3, 6, 12],
            agg_funcs=["mean", "std", "min", "max"]
        )
    )

### Temporal Features

In [ ]:
# features.append(
#     engineer_features.TemporalFeatures
#     (
#         column="timestamp",
#         frequency="24h"
#     )
# )

# print("✓ Temporal features added")

### Fourier Features (Seasonal Patterns)

In [ ]:
# # Add Fourier features for capturing seasonal patterns in the data
# # These are useful for recurring patterns like yearly seasonality
# features.append(
#     engineer_features.FourierFeatures
#     (
#         column="timestamp",
#         max_value=365,  # Yearly seasonality (365 daily observations per year)
#         n_fourier_terms=4  # 4 sine and 4 cosine terms
#     )
# )

# print("✓ Fourier features added (yearly seasonality with 4 terms)")

# Feature Definition

In [43]:
from timeseries_forecast.config.feature_config import FeatureConfig 
from timeseries_forecast.data_prep.data_resample import resample_dataframe_with_aggregations
import pandas as pd

def define_features(df: pd.DataFrame, forecast_horizon: int):
    feature_engineer = engineer_features.FeatureEngineeringConfig(features=features)
    features_df, added_features = feature_engineer.create_features(df, forecast_horizon)

    #this line adds contemp eth log return as feature
    #added_features.extend(feature_name_eth_log_ret)

    feat_config = FeatureConfig(
        date="timestamp",
        target="BTC_Close_log_ret",
        continuous_features=added_features,
        categorical_features=[],
        boolean_features=[],
        index_cols=["type","timestamp"],
        exogenous_features=[],
    )

    return feat_config, features_df

In [46]:
#forecast horizon defined here 
feature_config, features_df = define_features(feature_df, forecast_horizon=1)

C:\Users\Acer\Desktop\OTH\FPP\fpp\src\timeseries_forecast\feature\autoregressive_features.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(**col_dict)
C:\Users\Acer\Desktop\OTH\FPP\fpp\src\timeseries_forecast\feature\autoregressive_features.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(**col_dict)
C:\Users\Acer\Desktop\OTH\FPP\fpp\src\timeseries_forecast\feature\autoregressive_features.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame

In [50]:
features_df.columns

Index(['Aluminium_Close', 'Aluminium_Open', 'Aluminium_High', 'Aluminium_Low',
       'Aluminium_Vol.', 'Aluminium_Mode', 'Copper_Close', 'Copper_Open',
       'Copper_High', 'Copper_Low',
       ...
       'Zinc_Vol._rolling_3_min', 'Zinc_Vol._rolling_3_max',
       'Zinc_Vol._rolling_6_mean', 'Zinc_Vol._rolling_6_std',
       'Zinc_Vol._rolling_6_min', 'Zinc_Vol._rolling_6_max',
       'Zinc_Vol._rolling_12_mean', 'Zinc_Vol._rolling_12_std',
       'Zinc_Vol._rolling_12_min', 'Zinc_Vol._rolling_12_max'],
      dtype='str', length=885)

In [51]:
df_train = features_df[features_df["Aluminium_Mode"] == "Train"].dropna()
df_val = features_df[features_df["Aluminium_Mode"] == "Val"]
df_test = features_df[features_df["Aluminium_Mode"] == "Test"]

In [52]:
from pathlib import Path

fin = PROJECT_ROOT / "data" / "fin"
fin.mkdir(parents=True, exist_ok=True)

output_path = fin / "features_train.csv"
df_train.to_csv(output_path, index=True)

output_path = fin / "features_test.csv"
df_test.to_csv(output_path, index=True)

output_path = fin / "features_val.csv"
df_val.to_csv(output_path, index=True)


print(f"Saved cleaned dataframe to: {output_path}")

Saved cleaned dataframe to: C:\Users\Acer\Desktop\OTH\FPP\fpp\data\fin\features_val.csv


In [ ]:
train_features, train_target, _ = feature_config.get_X_y(df_train)
val_features, val_target, _ = feature_config.get_X_y(df_val)
test_features, test_target, _ = feature_config.get_X_y(df_test)